In [ ]:
%pip -q install finlab TA-Lib


In [ ]:
"""Colab cell: API-key authentication and 2330 indicator checks."""
import contextlib, io, os, json, zipfile
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import finlab
from finlab import data
from IPython.display import display

# Key is never written into the notebook, report, or exported files.
_api_key = getpass('FinLab API Key（隱藏輸入）: ').strip()
if not _api_key:
    raise RuntimeError('尚未輸入 API Key')
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        finlab.login(_api_key)
except Exception:
    raise RuntimeError('API Key 登入失敗，請確認 Key 及帳戶權限') from None



FinLab API Key（隱藏輸入）: ··········


## 1. 最新季 ROE

In [ ]:
def roe_latest(stock_id):

    # 取得個股季ROE
    roe = data.get("fundamental_features:ROE稅後")
    roe = roe[str(stock_id)].sort_index().dropna()

    # 最新季ROE，單位：%
    roe_latest_pct = round(float(roe.iloc[-1]), 2)

    return roe_latest_pct

In [ ]:
roe_latest_pct = roe_latest("2330")
roe_latest_pct

11.39

## 2. 最新月營收年增率

In [ ]:
def revenue_yoy_latest(stock_id):

    # 取得個股月營收年增率
    revenue = data.get("monthly_revenue:去年同月增減(%)")
    revenue = revenue[str(stock_id)].sort_index().dropna()

    # 最新月份年增率，單位：%
    revenue_yoy_latest_pct = float(round(revenue.iloc[-1], 2))

    return revenue_yoy_latest_pct

In [ ]:
revenue_yoy_latest_pct = revenue_yoy_latest(stock_id="2330")
revenue_yoy_latest_pct

Daily usage: 644.5 / 5000 MB - monthly_revenue:去年同月增減(%)
Daily usage: 661.7 / 5000 MB - price:收盤價


np.float64(54.64)

## 3. 最新月營收月增率

In [ ]:
def revenue_mom_latest(stock_id):

    # 取得個股月營收月增率
    revenue = data.get("monthly_revenue:上月比較增減(%)")
    revenue = revenue[str(stock_id)].sort_index().dropna()

    # 最新月份月增率，單位：%
    revenue_mom_latest_pct = float(round(revenue.iloc[-1], 2))

    return revenue_mom_latest_pct

In [ ]:
revenue_mom_latest = revenue_mom_latest("2330")
revenue_mom_latest

-0.57

## 4. 收盤價

# 新增區段

In [ ]:
def close_latest(stock_id):

    # 取得個股未還原收盤價
    close = data.get("price:收盤價")
    close = close[str(stock_id)].sort_index()

    # 最新收盤價，單位：元／股
    close_twd = float(round(close.iloc[-1], 2))

    return close_twd

In [ ]:
close_latest = close_latest("2330")
close_latest

2385.0

## 5. 近5日報酬

In [ ]:
def return_5d(stock_id):

    # 取得個股還原收盤價
    p = data.get("etl:adj_close")
    p = p[str(stock_id)].sort_index()

    # 近5日報酬，單位：%
    return_5d_pct = float(round((p.iloc[-1] / p.iloc[-6] - 1) * 100, 2))

    return return_5d_pct

In [ ]:
return_5d =return_5d("2330")
return_5d

-2.25

## 6. 近20日報酬

In [ ]:
def return_20d(stock_id):

    # 取得個股還原收盤價
    p = data.get("etl:adj_close")
    p = p[str(stock_id)].sort_index()

    # 近20日報酬，單位：%
    return_20d_pct = float(round((p.iloc[-1] / p.iloc[-21] - 1) * 100, 2))

    return return_20d_pct

In [ ]:
return_20d_pct = return_20d("2330")
return_20d_pct

-3.83

## 7. 個股跟大盤近20相對報酬

In [ ]:
def excess_return_20d(stock_id):

    # 取得個股還原收盤價
    p = data.get("etl:adj_close")
    p = p[str(stock_id)].sort_index()


    # 取得大盤報酬指數
    b = data.get("benchmark_return:發行量加權股價報酬指數")
    b = b.squeeze("columns").sort_index()

    # 對齊交易日期
    b = b.reindex(p.index)

    # 個股近20日報酬
    return_20d_pct = (p.iloc[-1] / p.iloc[-21] - 1)*100

    # 大盤近20日報酬
    benchmark_return_20d_pct = (b.iloc[-1] / b.iloc[-21] - 1)*100

    # 近20日相對大盤報酬 四捨五入至小數第二位
    excess_return_20d_pp =  round(return_20d_pct - benchmark_return_20d_pct, 2)

    return excess_return_20d_pp

## 8. 5日均量／20日均量

In [ ]:
def volume_ma5_to_ma20(stock_id):

    # 取得個股成交股數
    volume = data.get("price:成交股數")
    volume = volume[str(stock_id)].sort_index()

    # 計算5日與20日平均成交股數
    volume_ma5 = volume.rolling(5).mean().iloc[-1]
    volume_ma20 = volume.rolling(20).mean().iloc[-1]

    # 均量比，單位：倍
    volume_ma5_to_ma20_ratio = float(round(volume_ma5 / volume_ma20, 2))

    return volume_ma5_to_ma20_ratio

In [ ]:
volume_ma5_to_ma20 = volume_ma5_to_ma20("2330")
volume_ma5_to_ma20

1.1

## 9. 外資近5日買賣超強度

In [ ]:
def foreign_net_buy_intensity_5d(stock_id):

    # 取得個股成交股數
    volume = data.get("price:成交股數")
    volume = volume[str(stock_id)].sort_index()

    # 取得外資買賣超股數
    foreign = data.get(
        "institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)"
    )
    foreign = foreign[str(stock_id)].sort_index()

    # 對齊交易日期
    foreign = foreign.reindex(volume.index)

    # 近5日外資買賣超股數／近5日成交股數，單位：%
    foreign_net_buy_intensity_5d_pct = float(round(
        foreign.rolling(5).sum().iloc[-1]
        / volume.rolling(5).sum().iloc[-1] * 100, 2
    ))

    return foreign_net_buy_intensity_5d_pct

In [ ]:
foreign_net_buy_intensity_5d =foreign_net_buy_intensity_5d("2330")
foreign_net_buy_intensity_5d

-40.05

## 10. 外資最新交易日買賣超股數

In [ ]:
def foreign_net_buy_1d(stock_id):

    # 取得外資買賣超股數
    foreign = data.get(
        "institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)"
    )
    foreign = foreign[str(stock_id)].sort_index()

    # 最新交易日買賣超，單位：股
    foreign_net_buy_1d_shares = float(foreign.iloc[-1])

    return foreign_net_buy_1d_shares

In [ ]:
foreign_net_buy_1d =foreign_net_buy_1d("2330")
foreign_net_buy_1d

-12694626.0

## 11. 投信近5日買賣超強度

In [ ]:
def trust_net_buy_intensity_5d(stock_id):

    # 取得個股成交股數
    volume = data.get("price:成交股數")
    volume = volume[str(stock_id)].sort_index()

    # 取得投信買賣超股數
    trust = data.get("institutional_investors_trading_summary:投信買賣超股數")
    trust = trust[str(stock_id)].sort_index()

    # 對齊交易日期
    trust = trust.reindex(volume.index)

    # 近5日投信買賣超股數／近5日成交股數，單位：%
    trust_net_buy_intensity_5d_pct = float(round(
        trust.rolling(5).sum().iloc[-1]
        / volume.rolling(5).sum().iloc[-1] * 100, 2
    ))

    return trust_net_buy_intensity_5d_pct

In [ ]:
trust_net_buy_intensity_5d =trust_net_buy_intensity_5d("2330")
trust_net_buy_intensity_5d

5.58

## 12. 投信最新交易日買賣超股數

In [ ]:
def trust_net_buy_1d(stock_id):

    # 取得投信買賣超股數
    trust = data.get("institutional_investors_trading_summary:投信買賣超股數")
    trust = trust[str(stock_id)].sort_index()

    # 最新交易日買賣超，單位：股
    trust_net_buy_1d_shares = float(trust.iloc[-1])

    return trust_net_buy_1d_shares

In [ ]:
trust_net_buy_1d =trust_net_buy_1d("2330")
trust_net_buy_1d

1349951.0

## 13. 20日平均成交金額

In [ ]:
def amount_ma20(stock_id):

    # 取得個股成交金額
    amount = data.get("price:成交金額")
    amount = amount[str(stock_id)].sort_index()

    # 20日平均成交金額，保留原始金額單位
    amount_ma20_twd = float(round(amount.rolling(20).mean().iloc[-1], 2))

    return amount_ma20_twd

In [ ]:
amount_ma20 =amount_ma20("2330")
amount_ma20

67641744447.3

## 14. 10日報酬波動率

In [ ]:
def volatility_10d(stock_id):

    # 取得個股還原收盤價
    p = data.get("etl:adj_close")
    p = p[str(stock_id)].sort_index()

    # 每日報酬
    daily_return = p / p.shift(1) - 1

    # 最近10日日報酬的樣本標準差，單位：%，未年化
    volatility_10d_daily_pct = float(round(
        daily_return.rolling(10).std(ddof=1).iloc[-1] * 100, 2
    ))

    return volatility_10d_daily_pct

In [ ]:
volatility_10d =volatility_10d("2330")
volatility_10d

0.89

## 15. 20日均線近5日變化率

In [ ]:
def ma20_change_5d(stock_id):

    # 取得個股還原收盤價
    p = data.get("etl:adj_close")
    p = p[str(stock_id)].sort_index()

    # 計算20日均線
    ma20 = p.rolling(20).mean()

    # 最新20日均線相較5個交易日前的變化，單位：%
    ma20_change_5d_pct = float(round(
        (ma20.iloc[-1] / ma20.iloc[-6] - 1) * 100, 2
    ))

    return ma20_change_5d_pct

In [ ]:
ma20_change_5d =ma20_change_5d("2330")
ma20_change_5d

-0.15

## 16. 20日均線乖離率

In [ ]:
def price_vs_ma20(stock_id):

    # 取得個股還原收盤價
    p = data.get("etl:adj_close")
    p = p[str(stock_id)].sort_index()

    # 最新20日均線
    ma20 = p.rolling(20).mean().iloc[-1]

    # 最新價格相對20日均線的乖離，單位：%
    price_vs_ma20_pct = float(round((p.iloc[-1] / ma20 - 1) * 100, 2))

    return price_vs_ma20_pct

In [ ]:
price_vs_ma20 = price_vs_ma20("2330")
price_vs_ma20

-4.03

## 17. RSI14

In [ ]:
def rsi_14(stock_id):

    # 使用還原價格計算14日RSI
    rsi = data.indicator("RSI", adjust_price=True, timeperiod=14)
    rsi = rsi[str(stock_id)].sort_index()

    # 最新RSI，範圍0～100
    rsi_14 = float(round(rsi.iloc[-1], 2))

    return rsi_14

In [ ]:
rsi_14 =rsi_14("2330")
rsi_14

37.91

# 新增

## 18. KD（5、3、3）

KD用來觀察收盤價在近期高低價區間的位置，數值通常介於0～100。本版使用TA-Lib STOCH，以還原價格計算：

- **RSV（未成熟隨機值）**＝（收盤價－近5日最低價）／（近5日最高價－近5日最低價）×100。
- **K值**＝RSV的3日簡單移動平均，反應較快。
- **D值**＝K值的3日簡單移動平均，變化較平滑。
- `matype=0`代表簡單移動平均（SMA），期間內每筆資料權重相同；與部分看盤軟體採遞迴平滑的KD可能不同。

判讀時，可觀察K、D的高低、變化方向、兩線差距與交叉情況。80以上常稱高檔、20以下常稱低檔，但高檔不一定立刻下跌，低檔也不一定立刻上漲。

### 回傳數值與時間脈絡（精簡6欄）

| 欄位 | 說明 |
|---|---|
| `k_5` | 最新K值，觀察目前高低位置 |
| `d_5` | 最新D值，搭配K判斷目前兩線關係 |
| `k_change_3d` | 最新K減3個交易日前K；正值表示上升、負值表示下降，單位為指標點數 |
| `kd_above_d_days` | K連續高於D的交易日數；相等或缺值即中斷 |
| `kd_below_d_days` | K連續低於D的交易日數；相等或缺值即中斷 |
| `kd_cross_days_ago_20d` | 最近20個交易日內，最近一次交叉距今幾個交易日，不分方向；0＝今天、1＝前一交易日、-1＝這20日未發生 |

向上交叉指前一日K≤D、最新一天K>D；向下交叉指前一日K≥D、最新一天K<D。前一日相等、當日嚴格超越也算交叉，當日剛好相等則不算。合併欄位只記錄最近一次交叉時間，不保留歷史交叉方向。

目前K在D上方時，`kd_above_d_days`大於0；在D下方時，`kd_below_d_days`大於0。兩者皆為0代表目前原始K與D相等，不推定交叉方向。若交叉距今天數為0，可用哪一個連續天數大於0辨認今日交叉方向。

連續天數依可取得歷史往回計算，不限於20日；事件距今天數只搜尋最近20個交易日。-1不表示歷史上從未交叉。

完整輸出至少需要最近21筆有效K、D資料；不足或缺漏時報錯，不跳過缺值。所有計算先使用原始精度，輸出才四捨五入至小數第二位。K、D四捨五入後可能看似相等，此時請依連續天數判斷原始兩線關係。


In [ ]:
def kd_latest(stock_id):
    import pandas as pd

    # KD觀察收盤價在近期高低價區間的位置，數值通常介於0～100。
    # RSV＝（收盤價－近5日最低價）／（近5日最高價－近5日最低價）×100。
    # K＝RSV的3日簡單平均；D＝K的3日簡單平均；K反應較快，D較平滑。
    # 本版使用SMA平滑，與部分看盤軟體的遞迴平滑KD可能不同。
    # 80以上常稱高檔、20以下常稱低檔，但不代表即將反轉。
    k, d = data.indicator(
        "STOCH", adjust_price=True,
        fastk_period=5,
        slowk_period=3, slowk_matype=0,
        slowd_period=3, slowd_matype=0
    )
    k = k[str(stock_id)].sort_index()
    d = d[str(stock_id)].reindex(k.index)

    # 最近20個交易日的交叉需21筆完整資料，不跳過缺值。
    if len(k) < 21 or pd.concat([k, d], axis=1).iloc[-21:].isna().any().any():
        raise ValueError("最近21個交易日的KD資料不足，無法完整計算近期交叉")

    # 近3日變化：最新K－3個交易日前K，單位為指標點數。
    k_change_3d = k.iloc[-1] - k.iloc[-4]

    # 從最新一天往回計算K高於／低於D的連續交易日數。
    # 使用原始精度；相等或缺值就中斷，不額外限制為20日。
    gap = k - d
    above_d_days = 0
    below_d_days = 0
    for value in gap.iloc[::-1]:
        if pd.isna(value) or value <= 0:
            break
        above_d_days += 1
    for value in gap.iloc[::-1]:
        if pd.isna(value) or value >= 0:
            break
        below_d_days += 1

    # 合併向上／向下交叉，只保留最近一次交叉距今天數。
    # 前日相等、今日嚴格超越也算交叉；今日剛好相等不算。
    cross_up = (k.shift(1) <= d.shift(1)) & (k > d)
    cross_down = (k.shift(1) >= d.shift(1)) & (k < d)
    crossed = cross_up | cross_down
    cross_days_ago = -1
    for days_ago in range(20):
        if crossed.iloc[-1 - days_ago]:
            cross_days_ago = days_ago
            break

    # 輸出才四捨五入；連續天數可辨認四捨五入後看似相等的兩線關係。
    return {
        "k_5": round(float(k.iloc[-1]), 2),  # 最新K
        "d_5": round(float(d.iloc[-1]), 2),  # 最新D
        "k_change_3d": round(float(k_change_3d), 2),  # K近3日變化
        "kd_above_d_days": above_d_days,  # K連續高於D的交易日數
        "kd_below_d_days": below_d_days,  # K連續低於D的交易日數
        "kd_cross_days_ago_20d": cross_days_ago  # 0＝今天；1＝前一交易日；-1＝近20日未交叉
    }


In [ ]:
kd_result = kd_latest("2330")
kd_result


{'k_5': 31.21,
 'd_5': 48.25,
 'k_5_previous': 52.93,
 'd_5_previous': 50.98,
 'kd_cross_up': 0,
 'kd_cross_down': 1}

## 19. MACD（6、13、5）

使用還原價格計算。EMA是指數移動平均，近期價格權重較高；DIF＝6日EMA－13日EMA；訊號線＝DIF的5日EMA；柱狀值＝DIF－訊號線，沿用TA-Lib原值，不乘2。

### 回傳數值與時間脈絡（精簡6欄）

| 欄位 | 說明 |
|---|---|
| `macd_dif` | 最新DIF；正值表示6日EMA高於13日EMA，負值則相反 |
| `macd_hist` | 最新柱狀值；正值表示DIF在訊號線上方，負值則在下方 |
| `macd_hist_change_3d` | 最新柱狀值減3個交易日前柱狀值；正值表示增加，負值表示減少 |
| `macd_hist_increasing_days` | 柱狀值連續增加的相鄰交易日變化次數；-3變-2也算增加 |
| `macd_hist_decreasing_days` | 柱狀值連續減少的相鄰交易日變化次數；持平或缺值即中斷 |
| `macd_cross_days_ago_20d` | 最近20個交易日內，最近一次DIF與訊號線交叉距今幾個交易日，不分方向；0＝今天、1＝前一交易日、-1＝這20日未發生 |

### 搭配判讀一：DIF＋柱狀值，區分目前均線關係與動能位置

| DIF | 柱狀值 | 可直接描述的狀態 |
|---|---|---|
| 正 | 正 | 6日EMA高於13日EMA，DIF也高於訊號線 |
| 正 | 負 | 6日EMA仍高於13日EMA，但DIF已低於訊號線，留意動能轉弱 |
| 負 | 正 | 6日EMA仍低於13日EMA，但DIF已高於訊號線，可能出現改善跡象 |
| 負 | 負 | 6日EMA低於13日EMA，DIF也低於訊號線；是否繼續轉弱仍需看柱狀值變化 |

DIF正負只反映目前均線關係，不能單憑正負判定差距正在擴大或縮小。DIF向上穿越訊號線時仍可能低於0，不能把訊號線交叉與零軸穿越混為一談。

### 搭配判讀二：柱狀值＋近3日變化，觀察正在改善或減弱

| 最新柱狀值 | 近3日變化 | 判讀 |
|---|---|---|
| 正 | 正 | 目前DIF高於訊號線，差值比3個交易日前更高；若期間未跨零，可描述為正差距擴大 |
| 正 | 負 | 目前仍為正，但比3個交易日前降低，正差距縮小 |
| 負 | 正 | 目前仍為負，但比3個交易日前提高，負差距縮小 |
| 負 | 負 | 目前DIF低於訊號線，差值比3個交易日前更低；若期間未跨零，可描述為負差距擴大 |

近3日變化只比較起點與終點，不表示每天都往同一方向；若期間跨零，須另外搭配交叉時間判讀，不能假設整段都在零軸同一側。

### 搭配判讀三：近3日變化＋連續增減天數，確認延續性

- 近3日增加，且連續增加3天：這3次相鄰交易日變化均為增加。
- 近3日增加，但目前連續減少2天：3日合計仍改善，最近兩天卻已轉弱。
- 柱狀值從-3、-2、-1逐日增加，仍是負柱，不能當成已經轉正；正柱逐日減少，也不表示已轉負。
- 連續天數由最新一天往回計算，依可取得歷史為限，不額外截在20日；持平或缺值中斷。

### 搭配判讀四：交叉距今天數＋目前柱狀值，辨識事件新舊

- 0表示今天才交叉，1表示前一交易日交叉，可區分剛改變的兩線關係與已維持一段時間的關係。
- -1只表示最近20個交易日沒有交叉，不表示沒有趨勢或歷史上從未交叉。
- 精簡版只保留最近一次交叉時間，不保留歷史交叉方向。柱狀值正負描述的是目前位置，不應一律當作歷史交叉方向；目前剛好等於0或四捨五入成0時，尤其不能反推方向。

例如DIF=-2.50、柱狀值=-0.30、近3日變化=+0.80、連續增加3天、近20日無交叉，可描述為：「6日EMA仍低於13日EMA，DIF仍在訊號線下方；但柱狀值連續3日增加，負差距縮小，出現改善跡象，尚未發生近期向上交叉。」

### 計算口徑與判讀限制

- 完整輸出至少需要最近21筆有效DIF、訊號線與柱狀值；不足或缺漏時報錯，不跳過缺值，也不當成沒有交叉。
- 向上交叉＝前日DIF≤訊號線、當日DIF>訊號線；向下交叉則相反。前日相等、當日嚴格超越也算交叉，當日剛好相等則不算。
- 所有計算使用原始精度，輸出時才取小數第二位。數值顯示0.00可能只是接近0，不能據此確定原始值正好等於0。
- DIF、柱狀值與其變化皆為價格差，不是百分比；不同股價的股票不能直接比較絕對數值強弱，跨股票比較需另行標準化。
- 這6欄提供目前位置、近期變化、延續性與交叉時間，不涵蓋背離、DIF本身斜率或歷史交叉方向。MACD是歷史價格推導的證據，不保證未來漲跌，盤整時可能反覆交叉，應搭配價格、成交量及其他資料。


In [ ]:
def macd_latest(stock_id):
    import pandas as pd

    # EMA＝指數移動平均，近期價格權重較高。
    # DIF＝6日EMA－13日EMA；正值表示6日EMA高於13日EMA。
    # signal（訊號線）＝DIF的5日EMA；hist（柱狀值）＝DIF－訊號線，不乘2。
    # DIF觀察均線關係，hist觀察DIF相對訊號線的位置；並非獨立買賣指令。
    dif, signal, hist = data.indicator(
        "MACD", adjust_price=True,
        fastperiod=6, slowperiod=13, signalperiod=5
    )
    dif = dif[str(stock_id)].sort_index()
    signal = signal[str(stock_id)].reindex(dif.index)
    hist = hist[str(stock_id)].reindex(dif.index)

    # 最近20日的交叉需要21筆完整資料，不跳過缺值。
    if len(dif) < 21 or pd.concat([dif, signal, hist], axis=1).iloc[-21:].isna().any().any():
        raise ValueError("最近21個交易日的MACD資料不足，無法完整計算近期交叉")

    # 總變化看方向；連續天數看變化是否延續，兩者搭配使用。
    hist_change_3d = hist.iloc[-1] - hist.iloc[-4]
    hist_change = hist.diff()
    hist_increasing_days = 0
    hist_decreasing_days = 0
    for change in hist_change.iloc[::-1]:
        if pd.isna(change) or change <= 0:
            break
        hist_increasing_days += 1
    for change in hist_change.iloc[::-1]:
        if pd.isna(change) or change >= 0:
            break
        hist_decreasing_days += 1

    # 合併向上／向下穿越訊號線，只記最近一次距今天數，不是DIF穿越零軸。
    # 前日相等、今日嚴格超越也算；今日剛好相等不算。
    cross_up = (dif.shift(1) <= signal.shift(1)) & (dif > signal)
    cross_down = (dif.shift(1) >= signal.shift(1)) & (dif < signal)
    crossed = cross_up | cross_down
    cross_days_ago = -1
    for days_ago in range(20):
        if crossed.iloc[-1 - days_ago]:
            cross_days_ago = days_ago
            break

    # hist負但增加＝負差距縮小；hist正但減少＝正差距縮小。
    # 先用原始精度計算再四捨五入；輸出0.00可能只是很接近0，不宜反推交叉方向。
    return {
        "macd_dif": round(float(dif.iloc[-1]), 2),  # 最新DIF
        "macd_hist": round(float(hist.iloc[-1]), 2),  # 最新柱狀值，不乘2
        "macd_hist_change_3d": round(float(hist_change_3d), 2),  # 最新柱狀值－3個交易日前柱狀值
        "macd_hist_increasing_days": hist_increasing_days,  # 連續增加次數，負值變得較不負也算增加
        "macd_hist_decreasing_days": hist_decreasing_days,  # 連續減少次數，持平或缺值中斷
        "macd_cross_days_ago_20d": cross_days_ago  # 0＝今天；1＝前一交易日；-1＝近20日未交叉
    }


In [ ]:
macd_result = macd_latest("2330")
display(macd_result)


{'macd_dif': -68.5,
 'macd_signal': -56.07,
 'macd_hist': -12.42,
 'macd_dif_previous': -52.68,
 'macd_signal_previous': -49.86,
 'macd_hist_previous': -2.82,
 'macd_cross_up': 0,
 'macd_cross_down': 0}

## 20. 布林通道（10日、上下1.9個標準差）

使用還原價格，中軌是10日簡單移動平均，上軌＝中軌＋1.9個標準差，下軌＝中軌－1.9個標準差。

重點是三件事：**股價在通道哪裡、波動正在收斂或擴張、這個變化持續多久。** 精簡輸出為6個數值，保留價格位置與波動的時間脈絡。

### 回傳數值與時間脈絡（精簡6欄）

| 欄位 | 說明 |
|---|---|
| `bollinger_position_pct` | （還原收盤價－下軌）／（上軌－下軌）×100；0＝下軌、50＝中軌、100＝上軌 |
| `bollinger_bandwidth_pct` | （上軌－下軌）／中軌×100%，代表目前相對波動幅度 |
| `bollinger_bandwidth_change_3d_pp` | 最新帶寬減3個交易日前帶寬；正值表示較3日前擴張，負值表示收斂，單位為百分點 |
| `bollinger_contraction_days` | 從最新一天往回，帶寬連續縮小的相鄰交易日變化次數 |
| `bollinger_expansion_days` | 從最新一天往回，帶寬連續擴大的相鄰交易日變化次數 |
| `bollinger_bandwidth_percentile_60d` | 最近60筆（含今天）中，帶寬嚴格小於今日的筆數／60×100；越低表示相對自己近60日越窄 |

### 搭配判讀一：價格位置，觀察目前相對強弱

- 位置高於50：目前收盤價高於10日平均；低於50則低於10日平均。
- 越靠近100，價格越接近上軌；越靠近0，越接近下軌。這是相對近期區間的位置，不代表股票估值便宜或昂貴。
- 大於100表示目前收盤在上軌外，小於0表示在下軌外，程式不截斷數值。
- 上軌外不代表一定繼續上漲，下軌外不代表一定反彈。位置高低也不能單獨證明當天股價上漲或下跌，因為軌道本身會移動。

### 搭配判讀二：帶寬＋60日百分位，分辨目前波動與歷史位置

- 帶寬反映目前波動大小；百分位提供相對自身近60日的比較基準。
- 例如帶寬5%，只看這個值很難判斷對這檔股票是否偏窄；若百分位10，表示近60筆中約10%的帶寬嚴格小於目前，近期相對偏窄。
- 百分位低不代表未來一定擴張，也不能預測接下來的價格方向。
- 百分位採嚴格小於且包含今天，最高為59／60×100≈98.33，不會到100。若60筆帶寬都相同，百分位為0，不表示帶寬仍在下降，須搭配變化與連續天數。

### 搭配判讀三：近3日變化＋連續天數，確認收斂或擴張是否延續

- 近3日變化負、連續收斂3天：帶寬連續3次縮小，收斂具有近期延續性。
- 近3日變化正、連續擴張3天：帶寬連續3次擴大，近期波動持續增加。
- 近3日變化負，但目前連續擴張2天：3日合計仍較窄，最近兩天卻開始擴張，不能說仍每天持續收斂。
- 例如帶寬依序8.0%、7.2%、6.5%、5.9%，最新帶寬5.9%、近3日變化−2.1個百分點、連續收斂3天、連續擴張0天。
- 連續天數依可取得歷史往回計算，不限最近3日或60日；持平或缺值中斷，不跨越缺值。

### 搭配判讀四：價格位置＋帶寬變化，區分波動發生在哪一側

| 組合 | 可描述的狀態 |
|---|---|
| 價格靠近上軌、帶寬擴張 | 目前價格在近期區間偏高位置，波動增加；是否向上走強，仍須用報酬與成交量確認 |
| 價格靠近下軌、帶寬擴張 | 目前價格在近期區間偏低位置，波動增加；須搭配價格跌幅判斷下行風險 |
| 帶寬百分位偏低、連續收斂 | 波動相對近期偏低且繼續收斂，尚不能判定未來向上或向下 |

### 精簡範圍與計算限制

- 刪除軌道原始價格、前日數值、價格站在軌道上下的連續天數，以及突破旗標與距今天數，避免輸出過多欄位。
- 價格位置可表示目前是否位於軌道外，但不能判斷今天是否「剛突破」，或已連續在軌道外多久；精簡版不保留這些事件資訊。
- 完整輸出需要最近60筆有效帶寬與最新有效還原收盤價；不足或缺漏時報錯，不縮短期間或用0代替。中軌為0時無法計算帶寬，最新通道寬度為0時也會報錯。
- 計算與連續次數判斷均使用原始精度，輸出才取小數第二位。顯示0、50、100或近3日變化0.00，可能只是四捨五入結果，不能據此推定原始值剛好相等。
- 布林通道描述歷史價格位置與波動，後續方向交由Agent搭配價格報酬、成交量及其他證據判讀。


In [ ]:
def bollinger_latest(stock_id):
    import pandas as pd

    # 布林通道：中軌為10日簡單平均，上下軌各距中軌1.9個標準差。
    # 重點：價格在通道哪裡、波動正在擴大或收斂、是否已持續一段時間。
    upper, middle, lower = data.indicator(
        "BBANDS", adjust_price=True,
        timeperiod=10, nbdevup=1.9, nbdevdn=1.9, matype=0
    )

    # 價格與軌道都使用還原價格，避免不同口徑造成位置判斷錯誤。
    price = data.get("etl:adj_close")[str(stock_id)].sort_index()
    upper = upper[str(stock_id)].reindex(price.index)
    middle = middle[str(stock_id)].reindex(price.index)
    lower = lower[str(stock_id)].reindex(price.index)

    # 標準化帶寬＝（上軌－下軌）／中軌×100%，中軌為0時無法計算。
    bandwidth = (upper - lower) / middle.where(middle != 0) * 100
    if len(bandwidth) < 60 or bandwidth.iloc[-60:].isna().any():
        raise ValueError("最近60個交易日的帶寬資料不足，無法計算60日百分位")
    if pd.isna(price.iloc[-1]):
        raise ValueError("最新交易日的還原收盤價缺漏，無法計算通道位置")

    # 價格位置：0＝下軌、50＝中軌、100＝上軌；保留通道外的值。
    width = upper.iloc[-1] - lower.iloc[-1]
    if width <= 0:
        raise ValueError("最新布林通道寬度須大於0，無法計算價格位置")
    position_pct = (price.iloc[-1] - lower.iloc[-1]) / width * 100

    # 近3日帶寬變化，單位為百分點；不是帶寬變化率。
    bandwidth_change_3d = bandwidth.iloc[-1] - bandwidth.iloc[-4]
    bandwidth_change = bandwidth.diff()

    # 從最新一天往回計算連續縮小／擴大的變化次數；持平或缺值就中斷。
    contraction_days = 0
    expansion_days = 0
    for change in bandwidth_change.iloc[::-1]:
        if pd.isna(change) or change >= 0:
            break
        contraction_days += 1
    for change in bandwidth_change.iloc[::-1]:
        if pd.isna(change) or change <= 0:
            break
        expansion_days += 1

    # 近60筆（含今天）中，帶寬嚴格小於今天的比例，範圍0～98.33。
    # 越低表示相對自己近60日越窄；相等不算較小，不代表即將向上突破。
    bandwidth_percentile_60d = float((bandwidth.iloc[-60:] < bandwidth.iloc[-1]).mean() * 100)

    # 原始值先計算，輸出才取兩位小數。位置與帶寬都不是獨立買賣訊號。
    return {
        "bollinger_position_pct": round(float(position_pct), 2),  # 價格在通道中的相對位置
        "bollinger_bandwidth_pct": round(float(bandwidth.iloc[-1]), 2),  # 最新標準化帶寬，%
        "bollinger_bandwidth_change_3d_pp": round(float(bandwidth_change_3d), 2),  # 近3日變化，百分點
        "bollinger_contraction_days": contraction_days,  # 連續收斂次數
        "bollinger_expansion_days": expansion_days,  # 連續擴張次數
        "bollinger_bandwidth_percentile_60d": round(bandwidth_percentile_60d, 2)  # 帶寬相對自身近60日的位置
    }


In [ ]:
bollinger_result = bollinger_latest("2330")
display(bollinger_result)
